In [1]:
import os
from pathlib import Path
import json
from pprint import pprint
from collections import Counter

from colorama import Fore, Style

from datasets import load_dataset, DatasetDict
from huggingface_hub import dataset_info

from tqdm import tqdm

In [2]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)
LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"
print(DATASET_PATH)
print(LABEL_INFO_PATH)

/home/zelluzy/Desktop/pii_redaction_api/data/cleaned_ai4privacy_300k_pii
/home/zelluzy/Desktop/pii_redaction_api/data/label_info.json


In [3]:
ds_path = "ai4privacy/pii-masking-300k"
ds = load_dataset(ds_path)

In [4]:
# dataset card data
info = dataset_info(ds_path)
print(info.card_data)

language:
- en
- fr
- de
- it
- es
- nl
license: other
multilinguality:
- multilingual
size_categories:
- 100K<n<1M
source_datasets:
- original
task_categories:
- text-classification
- token-classification
- table-question-answering
- question-answering
- zero-shot-classification
- summarization
- feature-extraction
- text-generation
- translation
- fill-mask
- tabular-classification
- tabular-to-text
- table-to-text
- text-retrieval
- other
pretty_name: Ai4Privacy PII 300k Dataset
license_name: license.md
tags:
- legal
- business
- psychology
- privacy
- gdpr
- euaiact
- aiact
- pii
- sensitive
configs:
- config_name: default
  data_files:
  - split: train
    path: data/train/*.jsonl
  - split: validation
    path: data/validation/*.jsonl


In [5]:
# get dataset repr
print(repr(ds))
print("=" * 100, "\n")

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 177677
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 47728
    })
})



In [6]:
# get ds features
pprint(ds["train"].features)

{'id': Value('string'),
 'language': Value('string'),
 'mbert_bio_labels': List(Value('string')),
 'mbert_text_tokens': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'set': Value('string'),
 'source_text': Value('string'),
 'span_labels': Value('string'),
 'target_text': Value('string')}


In [7]:
for split in ds:
    print(split)

train
validation


In [8]:
# this dataset is multilingual. i will train on just the english examples for simplicity
print(f"length of full dataset: {sum([len(ds[split]) for split in ds]):,}")
ds = ds.filter(lambda x: x["language"] == "English")
print(f"English only ds length: {sum([len(ds[split]) for split in ds]):,}")

length of full dataset: 225,405


Filter:   0%|          | 0/177677 [00:00<?, ? examples/s]

Filter:   0%|          | 0/47728 [00:00<?, ? examples/s]

English only ds length: 37,854


In [9]:
# view single example from dataset
example = ds["train"][0]

for i in example:
    print(Fore.CYAN + f"{i}:")
    print(Style.RESET_ALL)
    pprint(f"{example[i]}")
    print("\n\n")

source_text:

('Subject: Group Messaging for Admissions Process\n'
 '\n'
 'Good morning, everyone,\n'
 '\n'
 'I hope this message finds you well. As we continue our admissions processes, '
 'I would like to update you on the latest developments and key information. '
 'Please find below the timeline for our upcoming meetings:\n'
 '\n'
 '- wynqvrh053 - Meeting at 10:20am\n'
 '- luka.burg - Meeting at 21\n'
 '- qahil.wittauer - Meeting at quarter past 13\n'
 '- gholamhossein.ruschke - Meeting at 9:47 PM\n'
 '- pdmjrsyoz1460 ')



target_text:

('Subject: Group Messaging for Admissions Process\n'
 '\n'
 'Good morning, everyone,\n'
 '\n'
 'I hope this message finds you well. As we continue our admissions processes, '
 'I would like to update you on the latest developments and key information. '
 'Please find below the timeline for our upcoming meetings:\n'
 '\n'
 '- [USERNAME] - Meeting at [TIME]\n'
 '- [USERNAME] - Meeting at [TIME]\n'
 '- [USERNAME] - Meeting at [TIME]\n'
 '- [USERNAME] 

There isn't a feature in this dataset to get the number of labels and their names.

In [10]:
train_mask_feat = ds["train"]["privacy_mask"]
print(next(iter(train_mask_feat))) # validate

[{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}, {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}, {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}, {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}, {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}, {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}, {'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'label': 'USERNAME'}, {'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'}, {'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}]


In [11]:
# count entities
train_entity_counter = Counter()
with tqdm(train_mask_feat) as pbar:
    for masks in pbar:
        train_entity_counter.update([mask["label"] for mask in masks])


100%|██████████| 29908/29908 [00:01<00:00, 18436.89it/s]


In [12]:
print(f"num_entities = {len(train_entity_counter.items())}")

print(Fore.CYAN + "sorted entities count for train_dataset:")
print(Style.RESET_ALL, end="")
for i in sorted(train_entity_counter.items()):
    print(i)

num_entities = 28
sorted entities count for train_dataset:
('BOD', 8461)
('BUILDING', 7383)
('CARDISSUER', 5)
('CITY', 7657)
('COUNTRY', 5875)
('DATE', 6780)
('DRIVERLICENSE', 8698)
('EMAIL', 9759)
('GEOCOORD', 703)
('GIVENNAME1', 7895)
('GIVENNAME2', 2102)
('IDCARD', 9609)
('IP', 8145)
('LASTNAME1', 9066)
('LASTNAME2', 2275)
('LASTNAME3', 734)
('PASS', 5637)
('PASSPORT', 9020)
('POSTCODE', 7458)
('SECADDRESS', 3117)
('SEX', 7496)
('SOCIALNUMBER', 9599)
('STATE', 7590)
('STREET', 7359)
('TEL', 7312)
('TIME', 14676)
('TITLE', 7560)
('USERNAME', 11115)


In [13]:
print(Fore.CYAN + "least common labels:")
for i in reversed(train_entity_counter.most_common()[-5:]):
    print(Style.RESET_ALL + str(i))

least common labels:
('CARDISSUER', 5)
('GEOCOORD', 703)
('LASTNAME3', 734)
('GIVENNAME2', 2102)
('LASTNAME2', 2275)


There are only 5 instances of 'CARDISSUER' which makes it effectively impossible to train on (unless i use upsampling). To keep F1 score reliable i will remove it.

In [14]:
# remove all bio tags that appear less than 50 times (CARDISSUER)
threshold = 50
# append keys to list to prevent:
# RuntimeError: dictionary changed size during iteration
keys_to_remove = []
for key, value in train_entity_counter.items():
    if value < threshold:
        keys_to_remove.append(key)
        
for key in keys_to_remove:
    del train_entity_counter[key]
    print(f"deleted tag {key}")
    
# print updated least common entity classes
print(Fore.CYAN + "updated least common entity classes:")
for i in reversed(train_entity_counter.most_common()[-5:]):
    print(Style.RESET_ALL + str(i))

deleted tag CARDISSUER
updated least common entity classes:
('GEOCOORD', 703)
('LASTNAME3', 734)
('GIVENNAME2', 2102)
('LASTNAME2', 2275)
('SECADDRESS', 3117)


In [15]:
entity_classes = list(train_entity_counter.keys())
print(entity_classes)

['USERNAME', 'TIME', 'DATE', 'LASTNAME1', 'LASTNAME2', 'EMAIL', 'SOCIALNUMBER', 'IDCARD', 'COUNTRY', 'BUILDING', 'STREET', 'CITY', 'STATE', 'POSTCODE', 'PASS', 'PASSPORT', 'TEL', 'DRIVERLICENSE', 'BOD', 'SEX', 'IP', 'SECADDRESS', 'LASTNAME3', 'GIVENNAME1', 'GIVENNAME2', 'TITLE', 'GEOCOORD']


In [16]:
# create bio tags mapped to ids
label2id = {"O": 0}
for i, entity in enumerate(entity_classes):
    b_id = 1 + i * 2
    i_id = b_id + 1
    entity_tags = {f"B-{entity}": b_id, f"I-{entity}": i_id}
    label2id.update(entity_tags)
print(f"{len(label2id)=}")
pprint(label2id, sort_dicts=False, indent=4)

len(label2id)=55
{   'O': 0,
    'B-USERNAME': 1,
    'I-USERNAME': 2,
    'B-TIME': 3,
    'I-TIME': 4,
    'B-DATE': 5,
    'I-DATE': 6,
    'B-LASTNAME1': 7,
    'I-LASTNAME1': 8,
    'B-LASTNAME2': 9,
    'I-LASTNAME2': 10,
    'B-EMAIL': 11,
    'I-EMAIL': 12,
    'B-SOCIALNUMBER': 13,
    'I-SOCIALNUMBER': 14,
    'B-IDCARD': 15,
    'I-IDCARD': 16,
    'B-COUNTRY': 17,
    'I-COUNTRY': 18,
    'B-BUILDING': 19,
    'I-BUILDING': 20,
    'B-STREET': 21,
    'I-STREET': 22,
    'B-CITY': 23,
    'I-CITY': 24,
    'B-STATE': 25,
    'I-STATE': 26,
    'B-POSTCODE': 27,
    'I-POSTCODE': 28,
    'B-PASS': 29,
    'I-PASS': 30,
    'B-PASSPORT': 31,
    'I-PASSPORT': 32,
    'B-TEL': 33,
    'I-TEL': 34,
    'B-DRIVERLICENSE': 35,
    'I-DRIVERLICENSE': 36,
    'B-BOD': 37,
    'I-BOD': 38,
    'B-SEX': 39,
    'I-SEX': 40,
    'B-IP': 41,
    'I-IP': 42,
    'B-SECADDRESS': 43,
    'I-SECADDRESS': 44,
    'B-LASTNAME3': 45,
    'I-LASTNAME3': 46,
    'B-GIVENNAME1': 47,
    'I-GIVEN

Since i will label my dataset with the "privacy_mask" feature, I will also need to remove CARDISSUER from privacy_masks.

In [17]:
# get full nested tree of pm
pprint(ds["train"].features["privacy_mask"])

List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')})


In [18]:
# count entities
val_entity_counter = Counter()
val_mask_feat = ds["validation"]["privacy_mask"]
with tqdm(val_mask_feat) as pbar:
    for masks in pbar:
        val_entity_counter.update([mask["label"] for mask in masks])
    
print(Fore.CYAN + "least common val labels:")
for i in reversed(val_entity_counter.most_common()[-5:]):
    print(Style.RESET_ALL + str(i))

100%|██████████| 7946/7946 [00:00<00:00, 17280.95it/s]

least common val labels:
('CARDISSUER', 1)
('LASTNAME3', 200)
('GEOCOORD', 216)
('GIVENNAME2', 539)
('LASTNAME2', 634)


In [19]:
# keep track of how many masks are removed, this should be 6 (5 from train 1 from val)
count_removed = 0

# remove cardissuer from dataset privacy mask
def remove_tag_from_pm(x, tag_names_to_remove: list=["CARDISSUER"]):
    pm: list[dict] = x["privacy_mask"]

    filtered = [d for d in pm if d["label"] not in tag_names_to_remove]
    if len(filtered) != len(pm):
        global count_removed # set to global so that i can update it in the function
        count_removed += (len(pm) - len(filtered))
    return {"privacy_mask": filtered}

cleaned_ds = ds.map(remove_tag_from_pm)

print(f"{count_removed=!r}")

count_removed=0


In [20]:
pprint(cleaned_ds["train"].features)

{'id': Value('string'),
 'language': Value('string'),
 'mbert_bio_labels': List(Value('string')),
 'mbert_text_tokens': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'set': Value('string'),
 'source_text': Value('string'),
 'span_labels': Value('string'),
 'target_text': Value('string')}


In [21]:
# remove all unnecessary columns before saving
columns_to_remove = [col for col in ds["train"].column_names if col not in ["source_text", "privacy_mask"]]
cleaned_ds = cleaned_ds.remove_columns(columns_to_remove)
pprint(cleaned_ds["train"].features)

{'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string')}


In [22]:
total_len = len(cleaned_ds["train"]) + len(cleaned_ds["validation"])
train_split = len(cleaned_ds["train"]) / total_len
val_split = len(cleaned_ds["validation"]) / total_len
print(f"train_split = {train_split * 100:.2f}%")
print(f"validation_split = {val_split * 100:.2f}%")

train_split = 79.01%
validation_split = 20.99%


I will split val dataset into validation and test split so that i can have a good representation of my models' performance.

In [23]:
val_test = cleaned_ds["validation"].train_test_split(test_size=0.5, seed=42)
new_val_ds = val_test["train"]
new_test_ds = val_test["test"]

new_val_split = len(new_val_ds) / total_len
new_test_split = len(new_test_ds) / total_len
print(f"train_split = {train_split * 100:.2f}%")
print(f"new validation split = {new_val_split * 100:.2f}%")
print(f"new test split = {new_test_split * 100:.2f}%")

train_split = 79.01%
new validation split = 10.50%
new test split = 10.50%


In [24]:
updated_and_cleaned_dataset = DatasetDict({
    "train": cleaned_ds["train"],
    "validation": new_val_ds,
    "test": new_test_ds,
})

In [25]:
pprint(updated_and_cleaned_dataset["train"].features)

{'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string')}


In [26]:
# make sure dataset doesn't contain cardissuer
entity_counter = Counter()
for split in updated_and_cleaned_dataset:
    with tqdm(updated_and_cleaned_dataset[split]["privacy_mask"]) as pbar:
        for masks in pbar:
            entity_counter.update([mask["label"] for mask in masks])
            
assert entity_counter.get("CARDISSUER") == None
assert entity_counter.get("USERNAME")

100%|██████████| 3973/3973 [00:00<00:00, 19516.55it/s]


In [27]:
id2label = {i:label for label, i in label2id.items()}
pprint(id2label)

{0: 'O',
 1: 'B-USERNAME',
 2: 'I-USERNAME',
 3: 'B-TIME',
 4: 'I-TIME',
 5: 'B-DATE',
 6: 'I-DATE',
 7: 'B-LASTNAME1',
 8: 'I-LASTNAME1',
 9: 'B-LASTNAME2',
 10: 'I-LASTNAME2',
 11: 'B-EMAIL',
 12: 'I-EMAIL',
 13: 'B-SOCIALNUMBER',
 14: 'I-SOCIALNUMBER',
 15: 'B-IDCARD',
 16: 'I-IDCARD',
 17: 'B-COUNTRY',
 18: 'I-COUNTRY',
 19: 'B-BUILDING',
 20: 'I-BUILDING',
 21: 'B-STREET',
 22: 'I-STREET',
 23: 'B-CITY',
 24: 'I-CITY',
 25: 'B-STATE',
 26: 'I-STATE',
 27: 'B-POSTCODE',
 28: 'I-POSTCODE',
 29: 'B-PASS',
 30: 'I-PASS',
 31: 'B-PASSPORT',
 32: 'I-PASSPORT',
 33: 'B-TEL',
 34: 'I-TEL',
 35: 'B-DRIVERLICENSE',
 36: 'I-DRIVERLICENSE',
 37: 'B-BOD',
 38: 'I-BOD',
 39: 'B-SEX',
 40: 'I-SEX',
 41: 'B-IP',
 42: 'I-IP',
 43: 'B-SECADDRESS',
 44: 'I-SECADDRESS',
 45: 'B-LASTNAME3',
 46: 'I-LASTNAME3',
 47: 'B-GIVENNAME1',
 48: 'I-GIVENNAME1',
 49: 'B-GIVENNAME2',
 50: 'I-GIVENNAME2',
 51: 'B-TITLE',
 52: 'I-TITLE',
 53: 'B-GEOCOORD',
 54: 'I-GEOCOORD'}


In [28]:
# create dict to save collected data
label_info = {
    "label2id": label2id,
    "id2label": id2label,
    "all_entities_counted": dict(entity_counter),
    "train_counted_entities": dict(train_entity_counter),
}

In [29]:
updated_and_cleaned_dataset.save_to_disk(DATASET_PATH)
# save as json
with open(LABEL_INFO_PATH, "w") as f:
    json.dump(label_info, f, indent=2)

Saving the dataset (0/1 shards):   0%|          | 0/29908 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3973 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3973 [00:00<?, ? examples/s]